# Notebook 02: Cleaning, missing data and univariate EDA

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 2, blocks 2 to 4

NexusHealth · Adamu Onu

You will turn the raw export into an analysis-ready dataset, document every decision in a cleaning log, decide how to handle missing data, and describe each variable. By the end you should be able to defend every row and every variable in your analysis set.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
})

# Find the course folder, wherever this notebook lives inside it.
ROOT <- normalizePath(getwd())
while (!dir.exists(file.path(ROOT, "data", "raw")) && dirname(ROOT) != ROOT) {
  ROOT <- dirname(ROOT)
}
stopifnot("Course folder not found. Open the whole course folder in VS Code." =
            dir.exists(file.path(ROOT, "data", "raw")))

DATA_RAW       <- file.path(ROOT, "data", "raw")
DATA_PROCESSED <- file.path(ROOT, "data", "processed")
DATA_REFERENCE <- file.path(ROOT, "data", "reference")
OUT_TABLES     <- file.path(ROOT, "outputs", "tables")
OUT_FIGURES    <- file.path(ROOT, "outputs", "figures")
for (folder in c(DATA_PROCESSED, OUT_TABLES, OUT_FIGURES)) dir.create(folder, recursive = TRUE, showWarnings = FALSE)

options(repr.plot.width = 7, repr.plot.height = 4.5, pillar.width = 120)
theme_set(theme_minimal(base_size = 12))
cat("Course folder found:", basename(ROOT), "\n")

# Part A: Data cleaning (block 2)

Cleaning follows five verbs: **audit, standardise, validate, deduplicate, derive**. Each rule is written in code (so it can be re-run) and recorded in a cleaning log (so a reader can judge it).

We read every column as **text** first. That way nothing is silently converted or hidden.

In [ ]:
raw <- read_csv(file.path(DATA_RAW, "primary_care_raw.csv"),
                col_types = cols(.default = "c"), na = character(), trim_ws = FALSE)
dictionary <- read_csv(file.path(DATA_RAW, "data_dictionary.csv"), show_col_types = FALSE)
dim(raw)
head(raw)

## A1. Audit: look before you change anything

**Categorical variables.** How many spellings of each category are there?

In [ ]:
for (col in c("clinic", "sex", "smoking_status")) {
  cat("\n", col, "\n")
  print(raw |> count(value = paste0("'", .data[[col]], "'")) |> arrange(desc(n)), n = Inf)
}

The quotation marks make stray spaces visible, such as `' North '`.

**Numeric variables.** Which entries cannot be read as numbers?

In [ ]:
NUMERIC <- c("age", "bmi", "systolic_bp", "diastolic_bp", "hba1c", "physical_activity_min",
             "medication_adherence", "diabetes", "followup_visits_6m")
for (col in NUMERIC) {
  bad <- raw[[col]][is.na(suppressWarnings(as.numeric(raw[[col]])))]
  if (length(bad) > 0) {
    tab <- table(paste0("'", bad, "'"))
    cat(sprintf("%-22s", col), paste(names(tab), tab, sep = ": ", collapse = ", "), "\n")
  }
}

**Duplicates.** Are any patients recorded more than once?

In [ ]:
cat("Exact duplicate rows:", sum(duplicated(raw)), "\n")
cat("Repeated patient IDs:", sum(duplicated(raw$patient_id)), "\n")
raw |> group_by(patient_id) |> filter(n() > 1) |> ungroup() |> arrange(patient_id) |> head(8)

**Audit summary.** List the problems you found. Which ones could change a result if ignored?

**Your answer:** _Double-click here and write your answer._

## A2. A cleaning log

Every rule gets one line in the log: what we did and how many values or rows it touched.

In [ ]:
cleaning_log <- tibble(step = integer(), rule = character(), n_affected = integer(), note = character())

log_step <- function(rule, n_affected, note = "") {
  cleaning_log <<- add_row(cleaning_log, step = nrow(cleaning_log) + 1L, rule = rule,
                           n_affected = as.integer(n_affected), note = note)
}

n_changed <- function(before, after) {
  # Count values that differ, treating missing as a value.
  sum(coalesce(as.character(before), "<NA>") != coalesce(as.character(after), "<NA>"))
}

clean <- raw

## A3. Standardise missing-value codes

In [ ]:
MISSING_TOKENS <- c("", "NA", "N/A", "Unknown")
n_tokens <- sum(sapply(clean, function(x) sum(x %in% MISSING_TOKENS)))
clean <- clean |> mutate(across(everything(), ~ if_else(.x %in% MISSING_TOKENS, NA_character_, .x)))
log_step("Recode blank, NA, N/A and Unknown to missing", n_tokens, "values")
cat("Values recoded to missing:", n_tokens, "\n")

## A4. Standardise categories

Trim spaces, use one capitalisation style, and expand abbreviations.

In [ ]:
before <- clean$clinic
clean <- clean |> mutate(clinic = str_to_title(str_trim(clinic)))
log_step("clinic: trim spaces and use title case", n_changed(before, clean$clinic))

before <- clean$sex
clean <- clean |> mutate(sex = str_to_title(str_trim(sex)),
                         sex = recode(sex, "F" = "Female", "M" = "Male"))
log_step("sex: trim, title case, F/M to Female/Male", n_changed(before, clean$sex))

print(count(clean, clinic))
print(count(clean, sex))

## Exercise 2A (part 1): Standardise smoking status

Apply the same idea to `smoking_status`. Then **check** that exactly three categories remain (`Never`, `Former`, `Current`) and add a line to the cleaning log.

**AI assist.** If you get stuck, ask Copilot: *"In R with dplyr and stringr, trim spaces and convert the text column smoking_status in tibble clean to title case. Then show the counts of each value."* Check that the counts add up to 653.

In [ ]:
before <- clean$smoking_status
clean <- clean |> mutate(smoking_status = ...)   # your rule here
log_step(...)                                    # describe the rule and count the changes

# Check: should print "Current" "Former" "Never"
sort(unique(na.omit(clean$smoking_status)))

## A5. Parse dates

Dates arrived in three formats: `2026-04-17`, `30/04/2026` and `Apr 06, 2026`. Is `02/04/2026` the 2nd of April or the 4th of February? Before assuming, check the evidence: if the first number is ever above 12, the format must be day first.

In [ ]:
slash <- clean$visit_date[str_detect(clean$visit_date, "/")]
parts <- str_split_fixed(slash, "/", 3)
cat("Slash dates:", length(slash), "| first part > 12:", sum(as.integer(parts[, 1]) > 12),
    "| second part > 12:", sum(as.integer(parts[, 2]) > 12), "\n")

The first part is often above 12 and the second never is, so these are day/month/year dates. We parse each format explicitly and stop if anything is left unparsed.

In [ ]:
original <- clean$visit_date
parse_quietly <- function(x, fmt) suppressWarnings(parse_date(x, format = fmt))
clean <- clean |>
  mutate(visit_date = coalesce(parse_quietly(original, "%Y-%m-%d"),
                               parse_quietly(original, "%d/%m/%Y"),
                               parse_quietly(original, "%b %d, %Y")))

stopifnot("Some dates could not be parsed" = !any(is.na(clean$visit_date)))
log_step("visit_date: parse three formats to ISO dates",
         sum(is.na(parse_quietly(original, "%Y-%m-%d"))),
         "non-ISO dates converted; slash dates are day/month/year")
range(clean$visit_date)

## A6. Convert numbers and validate ranges

Now the missing codes are gone, every numeric column should convert cleanly. We deliberately do **not** use a setting that hides failures: if a strange value remains, we want an error.

Then we apply the valid ranges from the data dictionary. Values outside the range become missing, and every change is logged.

In [ ]:
clean <- clean |> mutate(across(all_of(NUMERIC), as.numeric))   # warns if anything odd remains

VALID_RANGES <- list(   # copied from the data dictionary
  age = c(18, 85), bmi = c(10, 70), systolic_bp = c(70, 260), diastolic_bp = c(40, 160),
  hba1c = c(3, 20), physical_activity_min = c(0, 1500), medication_adherence = c(0, 100)
)
for (col in names(VALID_RANGES)) {
  low <- VALID_RANGES[[col]][1]; high <- VALID_RANGES[[col]][2]
  outside <- !is.na(clean[[col]]) & (clean[[col]] < low | clean[[col]] > high)
  if (any(outside)) {
    log_step(sprintf("%s: values outside %s to %s set to missing", col, low, high), sum(outside),
             paste("values seen:", paste(sort(unique(clean[[col]][outside])), collapse = ", ")))
    clean[[col]][outside] <- NA
  }
}
cleaning_log

## Exercise 2A (part 2): Defend a cleaning rule

Pick **one** rule from the log (for example the age rule). Write a four-sentence log entry: (1) what you saw, (2) the rule you applied, (3) how many values it affected, (4) what the rule could get wrong.

**Your answer:** _Double-click here and write your answer._

## A7. Remove duplicates

First exact duplicates (identical in every column). Then, for patients with more than one record, keep the **earliest** visit. Write the rule down *before* you look at which records it removes.

In [ ]:
n_before <- nrow(clean)
clean <- distinct(clean)
log_step("Remove exact duplicate rows", n_before - nrow(clean), "rows")

repeats <- clean |> group_by(patient_id) |> filter(n() > 1) |> ungroup() |> arrange(patient_id, visit_date)
print(repeats)

n_before <- nrow(clean)
clean <- clean |> arrange(patient_id, visit_date) |> distinct(patient_id, .keep_all = TRUE)
log_step("Keep the earliest record for repeated patient IDs", n_before - nrow(clean), "rows")
cat("Rows now:", nrow(clean), "| unique patients:", n_distinct(clean$patient_id), "\n")

## A8. Derive new variables

* `uncontrolled_bp` (the course-defined outcome): 1 if SBP is 140 or more **or** DBP is 90 or more; 0 if both are known and below those values; missing otherwise. Notice that a patient with SBP 150 and a missing DBP is still classified as uncontrolled.
* `adherence_10`: adherence divided by 10, so a one-unit change means 10 percentage points. This makes regression results easier to read.

In [ ]:
clean <- clean |>
  mutate(
    uncontrolled_bp = case_when(
      systolic_bp >= 140 | diastolic_bp >= 90         ~ 1,
      !is.na(systolic_bp) & !is.na(diastolic_bp)      ~ 0,
      .default = NA_real_
    ),
    adherence_10 = medication_adherence / 10
  )
log_step("Derive uncontrolled_bp (SBP >= 140 or DBP >= 90)", sum(!is.na(clean$uncontrolled_bp)),
         "teaching definition only")
log_step("Derive adherence_10 = medication_adherence / 10", sum(!is.na(clean$adherence_10)))

# A quick check of the derived outcome on a few rows
set.seed(1)
clean |> select(systolic_bp, diastolic_bp, uncontrolled_bp) |> slice_sample(n = 6)

## A9. Validate, then save

Assertions are tests that stop the notebook if a rule has been broken. They protect you when you re-run everything later, or when someone sends you an updated export.

In [ ]:
stopifnot(
  !anyDuplicated(clean$patient_id),
  all(na.omit(clean$clinic) %in% c("North", "East", "South", "West")),
  all(na.omit(clean$sex) %in% c("Female", "Male")),
  all(na.omit(clean$smoking_status) %in% c("Never", "Former", "Current")),
  all(na.omit(clean$uncontrolled_bp) %in% c(0, 1))
)
for (col in names(VALID_RANGES)) {
  stopifnot(all(between(na.omit(clean[[col]]), VALID_RANGES[[col]][1], VALID_RANGES[[col]][2])))
}

write_csv(clean, file.path(DATA_PROCESSED, "primary_care_clean.csv"), na = "")
write_csv(cleaning_log, file.path(OUT_TABLES, "cleaning_log.csv"))
cat("All checks passed. Saved", nrow(clean), "patients.\n")

# Part B: Missing data (block 3)

| Mechanism | Plain-language meaning | Example here |
|---|---|---|
| **MCAR** (missing completely at random) | Missingness has nothing to do with the patient | A form page lost at random |
| **MAR** (missing at random) | Missingness depends on things we *did* record | HbA1c less often measured in people without diabetes |
| **MNAR** (missing not at random) | Missingness depends on the missing value itself | People with poor adherence skip the adherence questions |

We can never prove which mechanism applies from the data alone. We can look for clues, choose a reasonable approach, and be honest about what it assumes.

## B1. How much is missing?

In [ ]:
missing_table <- clean |>
  summarise(across(everything(), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "n_missing") |>
  mutate(pct_missing = round(100 * n_missing / nrow(clean), 1)) |>
  arrange(desc(n_missing))
missing_table |> filter(n_missing > 0)

## B2. Looking for clues

Is HbA1c missing more often in some patients than others?

In [ ]:
clean |>
  group_by(diabetes) |>
  summarise(pct_hba1c_missing = round(100 * mean(is.na(hba1c)), 1))
clean |>
  group_by(clinic) |>
  summarise(pct_hba1c_missing = round(100 * mean(is.na(hba1c)), 1))

Now the main exposure. Do people with missing adherence look different on what we *did* record?

In [ ]:
clean |>
  group_by(adherence_missing = is.na(medication_adherence)) |>
  summarise(n = n(),
            mean_age = mean(age, na.rm = TRUE),
            pct_diabetes = 100 * mean(diabetes),
            mean_sbp = mean(systolic_bp, na.rm = TRUE)) |>
  mutate(across(where(is.numeric), ~ round(.x, 1)))

**What do these clues suggest?** For HbA1c and for adherence, say which mechanism looks plausible and why you cannot be certain.

**Your answer:** _Double-click here and write your answer._

## B3. Build the analysis set (complete-case analysis)

Our models need the outcome (SBP and DBP), the exposure (adherence) and the covariates (age, sex, BMI, diabetes, smoking). A **complete-case** analysis keeps only patients with all of these. It is simple and transparent, and it is a reasonable *starting point*. It can be biased if missingness depends on the outcome, and it always costs precision. Multiple imputation is the usual next step and needs further training.

We record how many patients are lost at each step, in the order the variables matter to the question.

In [ ]:
flow_steps <- list(
  "Unique patients after cleaning" = character(),
  "Outcome available (SBP and DBP)" = c("systolic_bp", "diastolic_bp"),
  "Exposure available (adherence)" = "medication_adherence",
  "Covariates available (age, sex, BMI, diabetes, smoking)" = c("age", "sex", "bmi", "diabetes", "smoking_status")
)
analysis <- clean
required <- character()
flow <- tibble(step = character(), n_remaining = integer(), n_excluded = integer())
for (label in names(flow_steps)) {
  required <- c(required, flow_steps[[label]])
  n_before <- nrow(analysis)
  analysis <- analysis |> drop_na(all_of(required))
  flow <- add_row(flow, step = label, n_remaining = nrow(analysis), n_excluded = n_before - nrow(analysis))
}
flow

## Exercise 2B (part 1): Who did we lose?

Compare patients **included** in the analysis set with those **excluded**, on age, diabetes and systolic BP. Use the `clean` data and mark who is in `analysis`.

In [ ]:
clean <- clean |> mutate(included = ...)   # TRUE if the patient_id is in the analysis set

clean |>
  group_by(included) |>
  summarise(...)

**Write one limitation sentence** that links the missing data to its likely consequence (for selection, bias or precision).

**Your answer:** _Double-click here and write your answer._

In [ ]:
write_csv(analysis, file.path(DATA_PROCESSED, "analysis_set.csv"), na = "")
write_csv(flow, file.path(OUT_TABLES, "analysis_set_flow.csv"))
cat("Saved analysis set:", dim(analysis), "\n")

# Part C: Univariate exploratory data analysis (block 4)

Univariate means one variable at a time. The right summary depends on the variable type:

| Variable type | Numeric summary | Graph |
|---|---|---|
| Continuous, roughly symmetric | Mean and standard deviation | Histogram |
| Continuous, skewed or with outliers | Median and interquartile range (IQR) | Histogram, boxplot |
| Categorical or binary | Count and percentage | Bar chart |

Always state the **denominator** (how many people the summary is based on).

## C1. Numeric variables

In [ ]:
numeric_vars <- c("age", "bmi", "systolic_bp", "diastolic_bp", "hba1c", "physical_activity_min",
                  "medication_adherence", "followup_visits_6m")

numeric_summary <- analysis |>
  select(all_of(numeric_vars)) |>
  pivot_longer(everything(), names_to = "variable") |>
  filter(!is.na(value)) |>
  group_by(variable = factor(variable, levels = numeric_vars)) |>
  summarise(n = n(), mean = mean(value), sd = sd(value), median = median(value),
            q1 = quantile(value, 0.25), q3 = quantile(value, 0.75),
            min = min(value), max = max(value)) |>
  mutate(across(where(is.numeric), ~ round(.x, 1)))
numeric_summary

Notice that HbA1c has a smaller `n` than the other variables. It was not needed for the analysis set, so its missing values are still there. Always report the `n` next to a summary.

In [ ]:
BRAND <- "#002366"

p_sbp <- ggplot(analysis, aes(systolic_bp)) +
  geom_histogram(bins = 25, fill = BRAND, colour = "white") +
  labs(title = "Systolic blood pressure", x = "SBP (mmHg)", y = "Patients")
p_adh <- ggplot(analysis, aes(medication_adherence)) +
  geom_histogram(bins = 25, fill = BRAND, colour = "white") +
  labs(title = "Medication adherence", x = "Adherence score (%)", y = "Patients")
print(p_sbp); print(p_adh)
ggsave(file.path(OUT_FIGURES, "fig_univariate_sbp.png"), p_sbp, width = 5, height = 3.8, dpi = 300)
ggsave(file.path(OUT_FIGURES, "fig_univariate_adherence.png"), p_adh, width = 5, height = 3.8, dpi = 300)

A boxplot is useful when a variable is skewed. Physical activity is a good example: most people do a little, a few do a lot.

In [ ]:
ggplot(analysis, aes(x = physical_activity_min)) +
  geom_boxplot(fill = "#DEE6EF", na.rm = TRUE) +
  labs(title = "Weekly physical activity", x = "Minutes per week") +
  theme(axis.text.y = element_blank())
cat("Mean:", round(mean(analysis$physical_activity_min, na.rm = TRUE), 1),
    "| Median:", median(analysis$physical_activity_min, na.rm = TRUE), "\n")

## C2. Categorical variables

In [ ]:
categorical_summary <- map_dfr(c("sex", "clinic", "smoking_status", "diabetes", "uncontrolled_bp"), function(col) {
  analysis |>
    count(level = as.character(.data[[col]])) |>
    mutate(variable = col, percent = round(100 * n / nrow(analysis), 1), .before = 1)
})
categorical_summary

In [ ]:
analysis |>
  mutate(smoking_status = factor(smoking_status, levels = c("Never", "Former", "Current"))) |>
  ggplot(aes(smoking_status)) +
  geom_bar(fill = BRAND) +
  labs(title = "Smoking status", x = NULL, y = "Patients")

## Exercise 2B (part 2): Your univariate evidence pack

Produce **one table and two figures** that describe BMI, clinic and the outcome `uncontrolled_bp`. Choose the summary and graph that suit each variable type. Save both figures to `outputs/figures`.

In [ ]:
# Table: summaries for BMI, clinic and uncontrolled_bp
...

# Figure 1
...

# Figure 2
...

**Write a five-sentence interpretation** of your evidence pack. Describe; do not explain causes.

**Your answer:** _Double-click here and write your answer._

## Key points

* Audit first, then standardise, validate, deduplicate and derive. Log every rule with a count.
* Let errors happen loudly (no silent coercion) and protect your rules with assertions.
* Missing data: describe how much, look for clues about why, choose an approach and state its assumption.
* A complete-case analysis is a transparent start, not a universal answer.
* Match the summary and the graph to the variable type, and always give the denominator.